# 04 — Benchmarking (FP16 vs quantized)


In [ ]:
import sys
sys.path.insert(0, "..")
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def benchmark_generation(model, tokenizer, prompt, n_tokens=50, n_runs=3, device="cpu"):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(
                **inputs,
                max_new_tokens=n_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return sum(times) / len(times)


tok = AutoTokenizer.from_pretrained("gpt2")
model_fp16 = AutoModelForCausalLM.from_pretrained("gpt2", torch_dtype=torch.float16)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model_fp16 = model_fp16.to(dev)
prompt = "The transformer architecture was introduced in 2017 and"
avg = benchmark_generation(model_fp16, tok, prompt, device=dev)
print(f"FP16 avg generate time: {avg:.3f}s on {dev}")


In [ ]:
import os
from pathlib import Path

def model_memory_gb(model):
    p = sum(x.numel() * x.element_size() for x in model.parameters())
    b = sum(x.numel() * x.element_size() for x in model.buffers())
    return (p + b) / (1024**3)

print(f"FP16 memory: {model_memory_gb(model_fp16):.3f} GB")
qdir = Path("..") / "quantized_gpt2"
if qdir.is_dir():
    print("Load quantized checkpoint with evaluate.py or replicate its load path to compare.")
else:
    print("Run notebook 03 first to create ../quantized_gpt2")
